In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"
df = pd.read_excel(url)

print("Original Dataset Shape:", df.shape)

Original Dataset Shape: (541909, 8)


In [ ]:
# Data Preprocessing
# Remove missing values
df.dropna(subset=['InvoiceNo', 'Description'], inplace=True)

# Convert InvoiceNo to string
df['InvoiceNo'] = df['InvoiceNo'].astype(str)

# Remove cancelled transactions (Invoice starting with 'C')
df = df[~df['InvoiceNo'].str.contains('C')]

In [ ]:
# Create basket (transaction matrix)
basket = (df
          .groupby(['InvoiceNo', 'Description'])['Quantity']
          .sum()
          .unstack()
          .fillna(0))

# Convert quantities into binary (0/1)
def encode_units(x):
    if x <= 0:
        return 0
    return 1

basket = basket.applymap(encode_units)

print("Basket Shape:", basket.shape)

# Optional: Reduce size (for faster execution in lab systems)
basket = basket.iloc[:5000, :]


/tmp/ipykernel_11370/2975916747.py:14: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  basket = basket.applymap(encode_units)


Basket Shape: (20610, 4207)


In [ ]:
# Apply Apriori
frequent_itemsets = apriori(basket, min_support=0.02, use_colnames=True)

print("\nFrequent Itemsets:")
print(frequent_itemsets.head())


Frequent Itemsets:
   support                               itemsets
0   0.0336     ( SET 2 TEA TOWELS I LOVE LONDON )
1   0.0270  (12 PENCILS SMALL TUBE RED RETROSPOT)
2   0.0216          (12 PENCILS SMALL TUBE SKULL)
3   0.0274     (3 HOOK PHOTO SHELF ANTIQUE WHITE)
4   0.0256   (3 PIECE SPACEBOY COOKIE CUTTER SET)


In [ ]:
# Generate Association Rules
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

print("\nAssociation Rules:")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head())


Association Rules:
                     antecedents                        consequents  support  \
0  (60 TEATIME FAIRY CAKE CASES)  (PACK OF 72 RETROSPOT CAKE CASES)   0.0280   
1  (ALARM CLOCK BAKELIKE ORANGE)       (ALARM CLOCK BAKELIKE GREEN)   0.0204   
2    (ALARM CLOCK BAKELIKE PINK)       (ALARM CLOCK BAKELIKE GREEN)   0.0210   
3   (ALARM CLOCK BAKELIKE GREEN)        (ALARM CLOCK BAKELIKE RED )   0.0352   
4    (ALARM CLOCK BAKELIKE RED )       (ALARM CLOCK BAKELIKE GREEN)   0.0352   

   confidence       lift  
0    0.595745   7.964501  
1    0.744526  13.838765  
2    0.567568  10.549583  
3    0.654275  12.729087  
4    0.684825  12.729087  


In [ ]:
# Sort rules by Lift
rules = rules.sort_values(by='lift', ascending=False)

print("\nTop Rules by Lift:")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))


Top Rules by Lift:
                                          antecedents  \
46                       (POPPY'S PLAYHOUSE BEDROOM )   
47                        (POPPY'S PLAYHOUSE KITCHEN)   
57  (PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY...   
58                   (PINK REGENCY TEACUP AND SAUCER)   
56  (GREEN REGENCY TEACUP AND SAUCER, PINK REGENCY...   
61  (REGENCY CAKESTAND 3 TIER, ROSES REGENCY TEACU...   
28                 (SINGLE HEART ZINC T-LIGHT HOLDER)   
14                  (HOT WATER BOTTLE I AM SO POORLY)   
23                   (PINK REGENCY TEACUP AND SAUCER)   
1                       (ALARM CLOCK BAKELIKE ORANGE)   

                                          consequents  support  confidence  \
46                        (POPPY'S PLAYHOUSE KITCHEN)   0.0220    0.753425   
47                       (POPPY'S PLAYHOUSE BEDROOM )   0.0220    0.733333   
57                  (GREEN REGENCY TEACUP AND SAUCER)   0.0202    0.961905   
58  (GREEN REGENCY TEACUP AND SAUCER, RO

In [ ]:
# Apriori Algorithm Demonstration

import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# Step 1: Create Transaction Dataset
dataset = [
    ['Milk', 'Bread', 'Butter'],
    ['Bread', 'Butter'],
    ['Milk', 'Bread'],
    ['Milk', 'Butter'],
    ['Bread', 'Butter'],
    ['Milk', 'Bread', 'Butter'],
    ['Milk', 'Bread'],
    ['Butter'],
    ['Milk', 'Butter'],
    ['Bread']
]

print("Transaction Dataset:")
for i, transaction in enumerate(dataset):
    print(f"T{i+1}: {transaction}")

# Step 2: Convert dataset into one-hot encoded DataFrame
te = TransactionEncoder()
te_array = te.fit(dataset).transform(dataset)
df = pd.DataFrame(te_array, columns=te.columns_)

print("\nOne-Hot Encoded Dataset:")
print(df)

# Step 3: Apply Apriori Algorithm
frequent_itemsets = apriori(df, min_support=0.3, use_colnames=True)

print("\nFrequent Itemsets:")
print(frequent_itemsets)

# Step 4: Generate Association Rules
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)

print("\nAssociation Rules:")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

Transaction Dataset:
T1: ['Milk', 'Bread', 'Butter']
T2: ['Bread', 'Butter']
T3: ['Milk', 'Bread']
T4: ['Milk', 'Butter']
T5: ['Bread', 'Butter']
T6: ['Milk', 'Bread', 'Butter']
T7: ['Milk', 'Bread']
T8: ['Butter']
T9: ['Milk', 'Butter']
T10: ['Bread']

One-Hot Encoded Dataset:
   Bread  Butter   Milk
0   True    True   True
1   True    True  False
2   True   False   True
3  False    True   True
4   True    True  False
5   True    True   True
6   True   False   True
7  False    True  False
8  False    True   True
9   True   False  False

Frequent Itemsets:
   support         itemsets
0      0.7          (Bread)
1      0.7         (Butter)
2      0.6           (Milk)
3      0.4  (Bread, Butter)
4      0.4    (Bread, Milk)
5      0.4   (Butter, Milk)

Association Rules:
  antecedents consequents  support  confidence      lift
0      (Milk)     (Bread)      0.4    0.666667  0.952381
1      (Milk)    (Butter)      0.4    0.666667  0.952381
